In [ ]:
import pandas as pd
import json
from collections import defaultdict

In [2]:
import os
os.getcwd()

'c:\\Users\\SiddheshLihe\\Transformer-LLM-Training-Tuning-Reasoning\\Unsloth_Chatmodel'

In [3]:
df = pd.read_csv("../Data/tbl_user_chat_202610081150.csv")

In [4]:
df

,id,user_id,message,response,context,created_at,updated_at,session_id
0,353,50,hey,Hi Prachi! 😊 It's always nice to chat with you...,NaN,2025-12-08 11:14:39.577,2025-12-08 11:14:39.577,191
1,354,50,hi,Hi Prachi! 😊 It's great to hear from you again...,NaN,2025-12-08 11:14:57.581,2025-12-08 11:14:57.581,192
2,361,50,chicken sausage,Hi Prachi! I see you've logged chicken sausage...,"{""meal_log_id"": ""4170"", ""meal_log"": {""meal_typ...",2025-12-09 05:30:32.410,2025-12-09 05:30:32.410,198
3,365,50,How much protein do I need daily?,"To determine your daily protein needs, let's c...",NaN,2025-12-09 05:42:25.940,2025-12-09 05:42:25.940,202
4,367,50,What to eat before heavy workout?,"For a heavy workout, it's essential to fuel yo...",NaN,2025-12-09 05:45:48.050,2025-12-09 05:45:48.050,204
...,...,...,...,...,...,...,...,...
25819,25988,27742,abhi ma ra weight kit na ho gaa,Agar aap abhi 75 kg hain aur aapka goal 68 kg ...,NaN,2026-10-07 17:40:24.421,2026-10-07 17:40:24.421,13271
25820,25989,27742,kit na din Lenge,Agar aapka current weight 75 kg hai aur aapka ...,NaN,2026-10-07 17:41:25.377,2026-10-07 17:41:25.377,13271
25821,25990,18319,Snack Plate,It looks like you had a Snack Plate with some ...,"{""meal_log_id"": ""236525"", ""meal_log"": {""meal_t...",2026-10-08 02:22:20.877,2026-10-08 02:22:20.877,13272
25822,25991,26945,What to eat before heavy workout?,"Before a heavy workout, it’s great to fuel you...",NaN,2026-10-08 05:32:36.086,2026-10-08 05:32:36.086,13273


In [10]:
df[df["message"].str.len() >=  2]

,id,user_id,message,response,context,created_at,updated_at,session_id
0,353,50,hey,Hi Prachi! 😊 It's always nice to chat with you...,NaN,2025-12-08 11:14:39.577,2025-12-08 11:14:39.577,191
1,354,50,hi,Hi Prachi! 😊 It's great to hear from you again...,NaN,2025-12-08 11:14:57.581,2025-12-08 11:14:57.581,192
2,361,50,chicken sausage,Hi Prachi! I see you've logged chicken sausage...,"{""meal_log_id"": ""4170"", ""meal_log"": {""meal_typ...",2025-12-09 05:30:32.410,2025-12-09 05:30:32.410,198
3,365,50,How much protein do I need daily?,"To determine your daily protein needs, let's c...",NaN,2025-12-09 05:42:25.940,2025-12-09 05:42:25.940,202
4,367,50,What to eat before heavy workout?,"For a heavy workout, it's essential to fuel yo...",NaN,2025-12-09 05:45:48.050,2025-12-09 05:45:48.050,204
...,...,...,...,...,...,...,...,...
25819,25988,27742,abhi ma ra weight kit na ho gaa,Agar aap abhi 75 kg hain aur aapka goal 68 kg ...,NaN,2026-10-07 17:40:24.421,2026-10-07 17:40:24.421,13271
25820,25989,27742,kit na din Lenge,Agar aapka current weight 75 kg hai aur aapka ...,NaN,2026-10-07 17:41:25.377,2026-10-07 17:41:25.377,13271
25821,25990,18319,Snack Plate,It looks like you had a Snack Plate with some ...,"{""meal_log_id"": ""236525"", ""meal_log"": {""meal_t...",2026-10-08 02:22:20.877,2026-10-08 02:22:20.877,13272
25822,25991,26945,What to eat before heavy workout?,"Before a heavy workout, it’s great to fuel you...",NaN,2026-10-08 05:32:36.086,2026-10-08 05:32:36.086,13273


In [11]:
df = df.dropna(subset=["message", "response"])
df = df[df["message"].str.len() > 2]
df = df[df["response"].str.len() > 5]

In [14]:
# 3. Name cleaning function
import re


def clean_response(text):
    text = str(text)
    
    # Common patterns काढून टाक
    # Hi Name! / Hello Name! / Hey Name!
    text = re.sub(r"(Hi|Hello|Hey|Hii|Hiii)\s+[A-Za-z]+[!.,]?\s*", r"\1! ", text, flags=re.IGNORECASE)
    
    # Extra spaces साफ कर
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

df["response_clean"] = df["response"].apply(clean_response)

In [15]:
df

,id,user_id,message,response,context,created_at,updated_at,session_id,response_clean
0,353,50,hey,Hi Prachi! 😊 It's always nice to chat with you...,NaN,2025-12-08 11:14:39.577,2025-12-08 11:14:39.577,191,Hi! 😊 It's always nice to chat with you. Are t...
2,361,50,chicken sausage,Hi Prachi! I see you've logged chicken sausage...,"{""meal_log_id"": ""4170"", ""meal_log"": {""meal_typ...",2025-12-09 05:30:32.410,2025-12-09 05:30:32.410,198,Hi! I see you've logged chicken sausage in you...
3,365,50,How much protein do I need daily?,"To determine your daily protein needs, let's c...",NaN,2025-12-09 05:42:25.940,2025-12-09 05:42:25.940,202,"To determine your daily protein needs, let's c..."
4,367,50,What to eat before heavy workout?,"For a heavy workout, it's essential to fuel yo...",NaN,2025-12-09 05:45:48.050,2025-12-09 05:45:48.050,204,"For a heavy workout, it's essential to fuel yo..."
5,372,430,how are you,I'm here and ready to help you with your healt...,NaN,2025-12-09 06:18:14.006,2025-12-09 06:18:14.006,153,I'm here and ready to help you with your healt...
...,...,...,...,...,...,...,...,...,...
25819,25988,27742,abhi ma ra weight kit na ho gaa,Agar aap abhi 75 kg hain aur aapka goal 68 kg ...,NaN,2026-10-07 17:40:24.421,2026-10-07 17:40:24.421,13271,Agar aap abhi 75 kg hain aur aapka goal 68 kg ...
25820,25989,27742,kit na din Lenge,Agar aapka current weight 75 kg hai aur aapka ...,NaN,2026-10-07 17:41:25.377,2026-10-07 17:41:25.377,13271,Agar aapka current weight 75 kg hai aur aapka ...
25821,25990,18319,Snack Plate,It looks like you had a Snack Plate with some ...,"{""meal_log_id"": ""236525"", ""meal_log"": {""meal_t...",2026-10-08 02:22:20.877,2026-10-08 02:22:20.877,13272,It looks like you had a Snack Plate with some ...
25822,25991,26945,What to eat before heavy workout?,"Before a heavy workout, it’s great to fuel you...",NaN,2026-10-08 05:32:36.086,2026-10-08 05:32:36.086,13273,"Before a heavy workout, it’s great to fuel you..."


In [18]:
# 4. Group by session
sessions = defaultdict(list)

for _, row in df.iterrows():
    sessions[row["session_id"]].append({
        "message": str(row["message"]).strip(),
        "response": row["response_clean"]
    })

In [19]:
# 5. Create ChatML format
training_data = []

system_prompt = """You are a friendly and knowledgeable Indian nutrition and fitness assistant. 
You give practical, easy-to-follow advice about food, protein, workouts, Indian meals, and healthy habits. 
Keep your replies warm, natural and helpful."""

In [20]:
for session_id, chats in sessions.items():
    messages = [{"role": "system", "content": system_prompt}]
    
    for chat in chats:
        messages.append({"role": "user", "content": chat["message"]})
        messages.append({"role": "assistant", "content": chat["response"]})
    
    if len(messages) >= 3:
        training_data.append({"messages": messages})

In [24]:
training_data[0:5]

[{'messages': [{'role': 'system',
    'content': 'You are a friendly and knowledgeable Indian nutrition and fitness assistant. \nYou give practical, easy-to-follow advice about food, protein, workouts, Indian meals, and healthy habits. \nKeep your replies warm, natural and helpful.'},
   {'role': 'user', 'content': 'hey'},
   {'role': 'assistant',
    'content': "Hi! 😊 It's always nice to chat with you. Are there any specific questions you have today about your meals or fitness routine? I'm here to help with anything you need on your journey to feeling healthier. How's everything going with your weight loss and daily routine? 🌿 Remember, this advice is tailored to your profile and goals."}]},
 {'messages': [{'role': 'system',
    'content': 'You are a friendly and knowledgeable Indian nutrition and fitness assistant. \nYou give practical, easy-to-follow advice about food, protein, workouts, Indian meals, and healthy habits. \nKeep your replies warm, natural and helpful.'},
   {'role': 

In [23]:
with open("../Data/train_data.json", "w", encoding="utf-8") as f:
    json.dump(training_data, f, ensure_ascii=False, indent=2)

print(f"Total conversations: {len(training_data)}")
print("Saved as train_data.json")

Total conversations: 12882
Saved as train_data.json


In [25]:
with open("../Data/train_data_20samples.json", "w", encoding="utf-8") as f:
    json.dump(training_data[0:20], f, ensure_ascii=False, indent=2)

print(f"Total conversations: {len(training_data[0:20])}")
print("Saved as train_data_20samples.json")

Total conversations: 20
Saved as train_data_20samples.json
